# 0 · Data preparation: load, join, clean

**Purpose.** This notebook builds the prepared dataset that all later steps use: one row per date and product group, with weather and Kiel Week attached, the weather-gap days removed, and every row marked as *training* or *validation*.

**What it does not do yet:** create new variables (features) such as holidays or weather-code classes. That comes in a separate notebook once those choices are made.

**Where the rules come from.** Every choice below was made by the team and is logged in `DECISIONS.md` (canvas block 7). The facts behind them are in `docs/data-fact-sheet.md`.

**Checkpoints.** After each step we run the same checks as `uv run pytest checks` (K1–K6, canvas block 8). If a check fails, the notebook stops with a plain message; do not change the check, find out what went wrong with the data.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make the shared preparation code (src/) and the checks (checks/) importable
ROOT = Path.cwd().parent if Path.cwd().name == "0_DataPreparation" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "checks")]

import bakery_prep as prep
import invariants as inv

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
GROUPS = {1: "Bread", 2: "Rolls", 3: "Croissants", 4: "Confectionery", 5: "Cakes", 6: "Seasonal bread"}

## 1. Load the raw data

Three files from the course repository. The raw data is only read, never changed.

- **Sales:** one row per date and product group, only for days with sales. `Umsatz` is revenue (currency not stated in the source, assumed €).
- **Weather:** one row per date with *observed* weather. It stands in for the forecast the manager would have (canvas block 6).
- **Kiel Week:** only the Kiel Week days.

In [2]:
sales = prep.load_sales()
weather = prep.load_weather()
kiwo = prep.load_kiwo()

pd.DataFrame({
    "rows": [len(sales), len(weather), len(kiwo)],
    "first date": [d["Datum"].min().date() for d in (sales, weather, kiwo)],
    "last date": [d["Datum"].max().date() for d in (sales, weather, kiwo)],
}, index=["sales", "weather", "kiel week"])

,rows,first date,last date
sales,9334,2013-07-01,2018-07-31
weather,2601,2012-01-01,2019-08-01
kiel week,72,2012-06-16,2019-06-30


**Checkpoint "load" (K1–K3).** Do the sales have the documented shape and unique keys (one row per date and group), is revenue always above 0, and are exactly the 38 known closed days missing from the calendar?

In [3]:
inv.check_sales_shape_and_keys(sales, raw_data_lines=9334)  # 9,334 = documented row count of the raw file
inv.check_revenue_positive(sales)
inv.check_calendar(sales)
print("K1-K3 passed")

K1-K3 passed


## 2. Join weather and Kiel Week

We attach weather and Kiel Week to each sales row by date. A *left join* keeps every sales row and only adds columns, so the row count must stay the same. Days that are not in the Kiel Week file get `KielerWoche = 0`.

Closed days (no sales at all) and days on which confectionery or seasonal bread was not offered have **no row** in the sales data. We leave it that way: we never add them as 0 revenue (decisions "Closed days not predicted" and "Not-offered days").

In [4]:
joined = prep.join(sales, weather, kiwo)

inv.check_join(joined, sales)  # K4
print(f"K4 passed: {len(joined):,} rows before and after the join, "
      f"{(joined['KielerWoche'] == 1).sum()} rows on Kiel Week days")
joined.head()

K4 passed: 9,334 rows before and after the join, 223 rows on Kiel Week days


,id,Datum,Warengruppe,Umsatz,Bewoelkung,Temperatur,Windgeschwindigkeit,Wettercode,KielerWoche
0,1307011,2013-07-01,1,148.828353,6.0,17.8375,15.0,20.0,0
1,1307021,2013-07-02,1,159.793757,3.0,17.3125,10.0,NaN,0
2,1307031,2013-07-03,1,111.885594,7.0,21.0750,6.0,61.0,0
3,1307041,2013-07-04,1,168.864941,7.0,18.8500,7.0,20.0,0
4,1307051,2013-07-05,1,171.280754,5.0,19.9750,12.0,NaN,0


## 3. Handle missing weather

How many values are missing per weather column after the join?

- `Wettercode` is missing mostly on calm, clear days, so a missing code means "no notable weather". We **keep** these rows; the code gets its own class later, when features are built.
- Missing cloud cover, temperature or wind: the team decided to **drop** these days instead of inventing values (decision "Weather gaps dropped").

In [5]:
joined[prep.WEATHER_REQUIRED + ["Wettercode"]].isna().sum().rename("missing values").to_frame()

,missing values
Bewoelkung,70
Temperatur,16
Windgeschwindigkeit,16
Wettercode,2325


In [6]:
prepared, removed = prep.drop_weather_gaps(joined)

inv.check_weather_drop(prepared, removed, joined)  # K5
print(f"K5 passed: removed {len(removed)} rows on {removed['Datum'].nunique()} dates")

(removed.groupby("Datum")
        .agg(rows=("Warengruppe", "size"),
             missing=("Bewoelkung", lambda s: "weather row" if removed.loc[s.index, "Temperatur"].isna().all() else "cloud cover"))
        .assign(weekday=lambda d: d.index.day_name()))

K5 passed: removed 70 rows on 12 dates


,rows,missing,weekday
Datum,,,
2016-12-11,6,weather row,Sunday
2017-10-04,5,weather row,Wednesday
2017-10-05,5,weather row,Thursday
2017-10-30,6,cloud cover,Monday
2017-11-01,6,cloud cover,Wednesday
2017-11-02,6,cloud cover,Thursday
2017-11-03,6,cloud cover,Friday
2017-11-04,6,cloud cover,Saturday
2017-11-05,6,cloud cover,Sunday


**What to notice:** 11 of these 12 dates are in the validation year, among them the first days of the 2017 seasonal-bread season. Validation therefore says little about the start of that season. The baseline and the model are scored on the same rows, so the comparison stays fair.

## 4. Mark training and validation

We split by **time**: everything before 2017-08-01 is training, 2017-08-01 to 2018-07-31 is validation (canvas block 4). This mimics real use, where the model only knows the past. The course's test year (2018-08-01 to 2019-07-31) is not in this data and is used once, at the very end.

In [7]:
train, val = prep.split(prepared)
inv.check_split(train, val, prepared)  # K6

prepared["split"] = "train"
prepared.loc[val.index, "split"] = "validation"
print("K6 passed")

(prepared.assign(group=prepared["Warengruppe"].map(GROUPS))
         .pivot_table(index=["Warengruppe", "group"], columns="split", values="Umsatz", aggfunc="size")
         [["train", "validation"]])

K6 passed


,split,train,validation
Warengruppe,group,,
1,Bread,1461,346
2,Rolls,1461,346
3,Croissants,1461,346
4,Confectionery,1408,346
5,Cakes,1461,346
6,Seasonal bread,235,47


**What to check:** confectionery (4) has fewer training rows than bread, rolls, croissants and cakes because of its not-offered days in 2013–2016. Seasonal bread (6) only appears from late October to early January.

## 5. First look: average revenue by weekday

A quick sanity check of the prepared data, using the training period only, so we don't look at the validation year before modelling. Weekdays are in calendar order.

In [8]:
(train.assign(weekday=pd.Categorical(train["Datum"].dt.day_name(), categories=WEEKDAYS, ordered=True),
              group=train["Warengruppe"].map(GROUPS))
      .pivot_table(index="weekday", columns="group", values="Umsatz", aggfunc="mean", observed=True)
      [list(GROUPS.values())]
      .round(1))

group,Bread,Rolls,Croissants,Confectionery,Cakes,Seasonal bread
weekday,,,,,,
Monday,132.5,373.4,152.0,82.6,256.4,63.0
Tuesday,118.1,358.7,145.1,77.9,264.0,66.6
Wednesday,117.8,356.1,145.5,76.3,270.8,65.6
Thursday,135.1,370.8,145.0,77.0,271.8,68.6
Friday,126.9,379.3,152.0,78.9,274.8,69.7
Saturday,145.8,488.8,204.3,85.7,318.3,76.4
Sunday,73.2,548.5,205.6,138.3,310.7,58.9


**What to notice:** the weekday pattern is **different for each group**, so a single weekly pattern for all groups would be wrong.
- **Rolls, croissants and cakes** sell most on Saturday and Sunday (rolls about 489 and 549 against about 356–379 on weekdays).
- **Bread** is *weakest* on Sunday (about 73 against 118–146 on other days); Saturday is its strongest day.
- **Confectionery** peaks on Sunday (about 138 against 76–86).
- **Seasonal bread** is lowest on Sunday as well (about 59 against 63–76).

These are training-period averages, not adjusted for season or holidays.

## 6. Save the prepared dataset

The file is the input for the next steps (dataset characteristics, baseline, model). It can be recreated at any time by running this notebook from top to bottom.

In [9]:
out_dir = ROOT / "0_DataPreparation" / "output"
out_dir.mkdir(exist_ok=True)
out_file = out_dir / "bakery_prepared.csv"
prepared.to_csv(out_file, index=False)

print(f"Saved {len(prepared):,} rows x {prepared.shape[1]} columns to {out_file.relative_to(ROOT)}")
print("Columns:", list(prepared.columns))

Saved 9,264 rows x 10 columns to 0_DataPreparation\output\bakery_prepared.csv
Columns: ['id', 'Datum', 'Warengruppe', 'Umsatz', 'Bewoelkung', 'Temperatur', 'Windgeschwindigkeit', 'Wettercode', 'KielerWoche', 'split']
